# Data modeling 6: warehouse architectures

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-06-warehouse-architectures.ipynb)

A star schema answers questions about one business, but a warehouse integrates many sources, keeps their raw
history, and serves many teams. How those pieces are arranged is the warehouse's architecture. Three schools
shape most designs: Kimball's dimensional bus, Inmon's normalized enterprise warehouse, and Data Vault. All of
them organize work into **layers** that each change the data in one way.

This lesson builds the shop's warehouse as layers in DuckDB schemas, builds a small Data Vault from the
customer and order sources, derives a dimension from it, and compares the approaches.

**Prerequisites:** [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb) and [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb). [Data modeling 5: normalized and operational models](data-modeling-05-normalized-and-operational-models.ipynb) helps with the Inmon section.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_customer_changes` (the CRM's audit log of segment and city changes), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order) and `src_products` (the product catalogue): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_customer_changes (customer_id int, changed_at timestamp, city varchar, segment varchar);
insert into src_customer_changes values  -- the CRM's audit log: one row each time a customer's details changed
    (101, '2025-11-02 00:00', 'Leeds', 'smb'),
    (102, '2025-11-20 00:00', 'York',  'smb'),
    (102, '2026-02-01 00:00', 'York',  'enterprise'),
    (103, '2025-12-05 00:00', 'Bath',  'enterprise'),
    (104, '2026-01-08 00:00', 'Hull',  'smb'),
    (104, '2026-02-15 00:00', 'Leeds', 'smb'),
    (105, '2026-01-19 00:00', 'Derby', 'smb'),
    (106, '2026-02-10 00:00', 'York',  'mid-market');
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
""")
con.sql("show tables").show()

## Three schools

| | Kimball | Inmon | Data Vault 2.0 |
|---|---|---|---|
| Core model | dimensional stars, integrated by conformed dimensions | normalized (3NF) enterprise warehouse | hubs, links and satellites |
| Built | bottom-up, one business process at a time | top-down, enterprise model first | incrementally, one source at a time |
| Marts | the stars are the warehouse | stars derived from the 3NF core | stars derived from the vault |
| Strength | fast to deliver, easy to query | one integrated version of the truth | absorbs new sources and changes; full audit trail |
| Cost | needs discipline to keep dimensions conformed | long, expensive start | many tables; a mart layer is always needed for querying |

In practice they are combined. A common modern layout lands raw data unchanged, integrates and historizes it
(in 3NF, Data Vault, or simply cleaned source-shaped tables), and publishes Kimball stars for consumption. The
star schema is almost always the layer people query; the schools disagree about what sits underneath it.

**Where it matters**
- Choosing a warehouse design, and recognizing which school an existing warehouse follows.

**Watch out for**
- Adopting Data Vault for a handful of stable sources: the overhead buys little until sources and changes are many.

## Layers

Each layer has one job, so a bug can be fixed by rebuilding one layer instead of reloading from the source:

| Layer | Medallion name | Job | Shape |
|---|---|---|---|
| **Raw / landing** | bronze | an exact, append-only copy of what each source sent | source-shaped, all strings or as-received |
| **Staging** | silver | type, rename, clean and deduplicate, one table per source table | source-shaped, clean |
| **Core / integration** | silver | integrate sources, assign surrogate keys, keep history | stars, 3NF or Data Vault |
| **Marts** | gold | serve one team or use: stars, aggregates, wide tables | dimensional |

The raw layer is the safety net: because it is never modified, every later layer can be rebuilt from it after
a logic fix. An **operational data store (ODS)** is a side branch: integrated, current-state data refreshed
frequently for operational lookups, with little history. A **data mart** is a subject-area slice (finance,
marketing); a *dependent* mart is built from the warehouse's conformed dimensions, while an *independent* mart
reads sources directly and soon disagrees with everyone else.

**Where it matters**
- Every warehouse and lakehouse, whatever its modeling school; dbt projects map these to folders (`staging`, `intermediate`, `marts`).

**Watch out for**
- Cleaning data in the raw layer: the original is lost, and with it the ability to replay.
- Marts that read staging or raw directly, skipping the conformed core: each one re-implements the business rules differently.

### Example

Build the layers as schemas. The raw order feed arrives as text, with a duplicate delivery of one record and a
status in mixed case, exactly as the source sent it.

In [ ]:
con.sql("""
create schema raw; create schema staging; create schema core; create schema mart;

create table raw.orders as                         -- as received: all text, plus a re-delivered record
select cast(order_id as varchar) as order_id, cast(customer_id as varchar) as customer_id,
       strftime(order_ts, '%Y-%m-%dT%H:%M:%S') as order_ts, upper(status) as status, now() as _loaded_at
from src_orders
union all
select '1006', '102', '2026-02-11T13:15:00', 'Shipped', now();

create table staging.orders as                     -- typed, standardized, one row per order
select distinct cast(order_id as int) as order_id, cast(customer_id as int) as customer_id,
       cast(order_ts as timestamp) as order_ts, lower(status) as status
from raw.orders;

create table core.fact_order as                    -- integrated: keyed, business rules applied
select order_id, customer_id, order_ts, status from staging.orders where status <> 'cancelled';

create table mart.orders_by_month as               -- shaped for one use
select date_trunc('month', order_ts)::date as month, count(*) as orders from core.fact_order group by all;
""")
con.sql("""
select 'raw.orders' as layer_table, count(*) as row_count from raw.orders
union all select 'staging.orders', count(*) from staging.orders
union all select 'core.fact_order', count(*) from core.fact_order
union all select 'mart.orders_by_month', count(*) from mart.orders_by_month
""").show()
con.sql("select order_id, status from raw.orders where order_id = '1006'").show()

**What to notice:** Raw keeps all 11 records exactly as sent, including the re-delivered order 1006 with its different casing.
Staging standardizes the case and removes the duplicate (10 orders); the core applies the business rule that
cancelled orders are not sales (9); the mart aggregates to 2 months. If the rule about cancelled orders changes,
only `core` and `mart` are rebuilt; the source is never asked for the data again.

## Data Vault: hubs, links and satellites

Data Vault splits every source entity into three kinds of insert-only table:

- A **hub** has one row per business key ever seen (`customer_id 101`), with a **hash key** computed from it,
  the time it was first loaded and the source that sent it. Hubs never change.
- A **link** has one row per relationship between business keys (order 1001 was placed by customer 101), keyed
  by the hash of the keys it joins. Links make every relationship many-to-many, so a relationship that changes
  cardinality later does not force a redesign.
- A **satellite** holds the descriptive attributes of a hub or link, with history: one row per change, with
  `load_ts`, `record_source` and a **hash diff** (a hash of all the attributes) so a load can detect a change
  by comparing one value.

Because keys are hashes of business keys, every table can be loaded in parallel without lookups, and because
everything is insert-only, the vault is a complete audit trail. The price is many tables and many joins, so
nobody queries the vault directly: **information marts** (stars) are built on top.

**Where it matters**
- Enterprises with many sources that change often, strict audit requirements, or several teams loading in parallel.

**Watch out for**
- Hashing business keys without standardizing them first (trim, case, type): ' 101' and '101' become different customers.
- Querying the raw vault for reports: build marts instead.

### Example

Build a hub, a link and a satellite from the customer and order sources. The CRM audit log arrives in two
batches: changes up to January 31, then the rest. Then rerun the second batch.

In [ ]:
con.sql("""
create table hub_customer (customer_hk varchar primary key, customer_id int, load_ts timestamp, record_source varchar);
create table hub_order    (order_hk varchar primary key, order_id int, load_ts timestamp, record_source varchar);
create table link_order_customer (link_hk varchar primary key, order_hk varchar, customer_hk varchar, load_ts timestamp, record_source varchar);
create table sat_customer (customer_hk varchar, load_ts timestamp, hash_diff varchar, city varchar, segment varchar, record_source varchar,
                           primary key (customer_hk, load_ts));
create macro hk(business_key) as md5(upper(trim(cast(business_key as varchar))));   -- standardize, then hash
""")

def load_customers(batch_ts):
    """Load the CRM changes made up to batch_ts into the hub and the satellite."""
    con.sql(f"""
insert into hub_customer
select distinct hk(customer_id), customer_id, timestamp '{batch_ts}', 'crm'
from src_customer_changes c
where c.changed_at <= timestamp '{batch_ts}' and hk(customer_id) not in (select customer_hk from hub_customer);

insert into sat_customer
select hk(c.customer_id), c.changed_at, md5(concat_ws('|', c.city, c.segment)), c.city, c.segment, 'crm'
from src_customer_changes c
where c.changed_at <= timestamp '{batch_ts}'
  and md5(concat_ws('|', c.city, c.segment)) is distinct from (     -- only rows that differ from the latest satellite row
      select arg_max(s.hash_diff, s.load_ts) from sat_customer s
      where s.customer_hk = hk(c.customer_id) and s.load_ts <= c.changed_at)
  and (hk(c.customer_id), c.changed_at) not in (select (customer_hk, load_ts) from sat_customer);
""")


for batch in ["2026-01-31", "2026-03-01", "2026-03-01"]:  # the last batch is a rerun
    load_customers(batch)
    counts = con.sql("select (select count(*) from hub_customer), (select count(*) from sat_customer)").fetchall()[0]
    print(f"after batch {batch}: hub_customer {counts[0]} rows, sat_customer {counts[1]} rows")

In [ ]:
con.sql("""
insert into hub_order select hk(order_id), order_id, now(), 'orders_app' from src_orders;
insert into link_order_customer
select hk(order_id || '|' || customer_id), hk(order_id), hk(customer_id), now(), 'orders_app' from src_orders;
""")
con.sql("""
select h.customer_id, left(h.customer_hk, 10) || '...' as customer_hk, s.load_ts::date as load_date,
       s.city, s.segment, left(s.hash_diff, 8) as hash_diff
from hub_customer h join sat_customer s using (customer_hk)
where h.customer_id in (102, 104)
order by h.customer_id, s.load_ts
""").show()

**What to notice:** The first batch loaded the 5 customers known by January 31 and one satellite row each. The second added customer
106 to the hub and three satellite rows: the two changes (102's segment, 104's city) and the new customer. The
rerun added nothing, because hubs insert only unseen keys and satellites only rows whose hash diff differs.
Customer 104's two satellite rows have different hash diffs because the city changed; that one comparison is
how the vault detects change in any number of columns.

### Example

An information mart: derive a Type 2 customer dimension from the hub and satellite. The validity range comes from
the satellite's load timestamps.

In [ ]:
con.sql("""
create table mart.dim_customer as
select row_number() over (order by h.customer_id, s.load_ts)                       as customer_sk,
       h.customer_id, s.city, s.segment,
       s.load_ts                                                                   as valid_from,
       coalesce(lead(s.load_ts) over (partition by h.customer_hk order by s.load_ts),
                timestamp '9999-12-31')                                            as valid_to
from hub_customer h join sat_customer s using (customer_hk);

select o.order_id, d.customer_id, d.segment, d.city
from link_order_customer l
join hub_order o using (order_hk)
join hub_customer h using (customer_hk)
join src_orders so on so.order_id = o.order_id
join mart.dim_customer d on d.customer_id = h.customer_id
 and so.order_ts >= d.valid_from and so.order_ts < d.valid_to
where d.customer_id in (102, 104)
order by o.order_id
""").show()

**What to notice:** The dimension built from the vault is the same Type 2 dimension lesson 3 built from the audit log, and the
point-in-time join gives 102's January order the `smb` segment and 104's February 3 order the Hull city. The
vault stored the history once, insert-only; the mart is a disposable view of it that can be rebuilt with new
rules at any time.

## Inmon's normalized core

In an Inmon warehouse the core layer is a normalized, integrated, historized model of the whole enterprise:
customers, products, orders and their relationships in 3NF, with every source mapped into one set of
definitions (lesson 5's normalization, plus history columns). Departmental stars are then derived from it.
Its strength is a single integrated model that new marts reuse. Its weakness is the large modeling effort
before the first report, and a core that must be redesigned when the business changes shape, which is the
problem Data Vault's insert-only hubs and links were designed to avoid.

**Where it matters**
- Organizations with a strong central data team and stable, well-understood subject areas (banks, insurers).

**Watch out for**
- Letting reports query the 3NF core directly: they need the same joins and rules over and over; that is what marts are for.

## Wide tables: one big table

Many modern teams add a **one big table** (OBT): the fact joined to all its dimensions and flattened into one
very wide table. Columnar engines read only the columns a query uses, so width is cheap; there are no joins at
query time, and BI tools and notebook users find it simple.

An OBT is a mart, not a replacement for the model. It is built *from* the star, which still owns the grain,
the keys, the history rules and the conformed dimensions. Without that, each wide table re-derives customer
segment or product category its own way. OBTs are also expensive to keep current when a dimension attribute
changes (every affected fact row is rewritten) and they fix one choice, such as "segment at the time of the
order", into the table.

**Where it matters**
- Dashboard backends, data science feature tables, and engines where joins are expensive.

**Watch out for**
- Building wide tables straight from staging: you lose conformed dimensions and the ability to answer a question the table was not built for.

### Example

Build a wide sales table from the vault-derived dimension and the order lines, then query it with no joins.

In [ ]:
con.sql("""
create table mart.obt_sales as
select l.order_id, l.line_no, so.order_ts, so.channel, so.status,
       d.customer_id, d.city as customer_city, d.segment as customer_segment,
       p.sku, p.product_name, p.category, p.department,
       l.qty as quantity, l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l
join src_orders so using (order_id)
join src_products p using (sku)
join mart.dim_customer d on d.customer_id = so.customer_id
 and so.order_ts >= d.valid_from and so.order_ts < d.valid_to
where so.status <> 'cancelled';

select customer_segment, department, count(*) as lines, sum(net_amount) as net_sales
from mart.obt_sales group by all order by all
""").show()

**What to notice:** The wide table answers department-by-segment without a single join, and its segment is the segment at the time of
the order, because it was built with the point-in-time join. That choice is now baked in: answering "by current
segment" needs another column or another table, whereas the star could answer both.

## Where modeling happens in a lakehouse and a mesh

The layers do not change in a **lakehouse**: bronze, silver and gold are tables in an open format (Delta, Iceberg)
on object storage instead of tables inside a warehouse, and the same modeling applies in each layer. Lesson 8 covers
the physical differences. In a **data mesh**, domain teams own their data products end to end, including their
models. What keeps a mesh coherent is the same thing that keeps a Kimball bus coherent: shared, conformed
dimensions (customer, product, date) and agreed metric definitions (lesson 9), owned by someone.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why should the raw layer never be cleaned in place?

**A.** It is the only exact copy of what the source sent. Keeping it unchanged means any later layer can be rebuilt after a logic fix without asking the source for the data again.

**Q2.** A Data Vault load computes `md5(customer_id)` without trimming or casting. What can go wrong?

**A.** The same customer sent as `101`, `' 101'` or `101.0` hashes differently and becomes several hub rows. Standardize business keys before hashing.

**Q3.** What is a hash diff used for?

**A.** Change detection in a satellite: compare one hash of all the attributes with the latest row's hash instead of comparing every column. If it differs, insert a new satellite row.

**Q4.** Your team wants to replace the star schema with one wide table per dashboard. What do you lose?

**A.** A single owner for grain, keys, history and conformed dimensions. Each wide table re-derives attributes its own way, attribute changes rewrite many rows, and questions the table was not built for cannot be answered.

**Q5.** When is Data Vault worth its overhead?

**A.** When there are many sources that change often, strict audit or reproducibility requirements, or many teams loading in parallel. For a few stable sources, cleaned staging tables plus stars are simpler.

## Key takeaways

- Kimball builds conformed stars bottom-up; Inmon builds a normalized enterprise core first; Data Vault builds insert-only hubs, links and satellites.
- Most warehouses combine them: raw, then an integration layer, then Kimball stars for querying.
- Raw is append-only and never cleaned, so every later layer can be rebuilt from it.
- Data Vault uses hash keys of standardized business keys and hash diffs to load in parallel and detect change; marts sit on top.
- A wide table is a convenient mart built from the star, not a substitute for it.

**Next:** Continue with [Data modeling 7: loading and late data](data-modeling-07-loading-and-late-data.ipynb).